# Tokens and Byte-Pair Encoding

Back in Part 1 we said an LLM predicts, one *token* at a time, what comes next. But what exactly is a token? In this part we open up that black box: why LLMs don't work directly on words or on individual characters, and how they build their vocabulary of tokens using an algorithm called **Byte-Pair Encoding (BPE)**. We'll write a tiny BPE trainer ourselves on a toy example, then look at how a real, production tokenizer splits actual text.

## Why not words, why not characters?

An LLM needs to turn text into a sequence of discrete symbols it can work with. Two obvious choices both have problems:

* **Whole words as symbols**: the vocabulary would need an entry for every word in every language, including rare words, typos, product names, etc. That's an enormous, ever-growing vocabulary, and the model still fails completely on any word it has never seen before.
* **Individual characters as symbols**: the vocabulary stays tiny (a few hundred symbols), and any string can be represented. But sequences become very long — a short sentence becomes dozens of symbols — which makes it harder and more expensive for the model to relate symbols that are far apart.

**Tokens** (subword units) are the middle ground: common words end up as a single token, while rare or long words get split into a handful of frequent pieces. This keeps the vocabulary at a reasonable size (tens of thousands of entries) while still being able to represent *any* string, since individual characters are always available as a fallback.

## Byte-Pair Encoding (BPE): the algorithm

BPE builds this middle-ground vocabulary automatically from a training corpus, by repeatedly merging the most frequent pair of adjacent symbols:

1. Start with a vocabulary made of every individual character (or byte) appearing in the corpus, and represent every word as a sequence of these characters.
2. Count how often every pair of adjacent symbols occurs across the whole corpus.
3. Take the single most frequent pair, merge it into one new symbol, and add that symbol to the vocabulary.
4. Repeat steps 2-3 a fixed number of times (this number of merges is chosen when training the tokenizer — real tokenizers typically do tens of thousands of merges).

The final vocabulary is the original characters plus every symbol created by a merge along the way. To tokenize new text, you just apply the same merges, in the same order, to it.

Let's see this in action on the small toy corpus from the original BPE paper (Sennrich et al., 2016): the words `low`, `lower`, `newest` and `widest`, each with a frequency, and a special `</w>` symbol marking the end of a word (so the tokenizer can tell "est" at the end of a word from "est" at the start of one).

In [ ]:
from collections import Counter

# each word is a tuple of symbols (starting as individual characters), with its frequency in the corpus
corpus = {
    ("l", "o", "w", "</w>"): 5,
    ("l", "o", "w", "e", "r", "</w>"): 2,
    ("n", "e", "w", "e", "s", "t", "</w>"): 6,
    ("w", "i", "d", "e", "s", "t", "</w>"): 3,
}

def get_pair_counts(corpus):
    """Count how often each pair of adjacent symbols occurs, across the whole corpus."""
    counts = Counter()
    for word, freq in corpus.items():
        for a, b in zip(word, word[1:]):
            counts[(a, b)] += freq
    return counts

def merge_pair(pair, corpus):
    """Replace every occurrence of `pair` by its merged symbol in every word."""
    a, b = pair
    merged = a + b
    new_corpus = {}
    for word, freq in corpus.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
                new_word.append(merged)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_corpus[tuple(new_word)] = freq
    return new_corpus

vocab = set(symbol for word in corpus for symbol in word)
num_merges = 8

for i in range(num_merges):
    pairs = get_pair_counts(corpus)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    corpus = merge_pair(best_pair, corpus)
    vocab.add("".join(best_pair))
    print(f"Merge {i + 1}: {best_pair} -> {best_pair[0] + best_pair[1]!r}  (seen {pairs[best_pair]} times)")

print("\nWords after all merges:")
for word, freq in corpus.items():
    print(f"  {word}  (frequency {freq})")

print(f"\nVocabulary size: {len(vocab)} symbols")

Watch how the merges build up meaningful pieces step by step: `e`+`s` merges into `es` (it appears in both "newest" and "widest"), then `es`+`t` into `est`, then `est`+`</w>` into a whole "est" ending. Separately, `l`+`o`+`w` merges into `low`. And because "newest" is frequent enough on its own, it eventually merges all the way into a single `newest</w>` token — while "widest", which is rarer, stays split into `w`, `i`, `d`, `est</w>`.

This is exactly the trade-off we described above: frequent words tend to become a single token, while rarer ones stay broken up into smaller, more frequent pieces.

## A real tokenizer: tiktoken

Real tokenizers are trained the exact same way, just on a much bigger corpus and with tens of thousands of merges instead of 8. When you call an LLM API, you don't train this yourself: the provider ships a fixed, pre-trained vocabulary and merge table, and you just reuse it to split your text into tokens.

[`tiktoken`](https://github.com/openai/tiktoken) is OpenAI's tokenizer library; it runs entirely locally (no API call needed) and lets us look at real tokenization in action.

In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

text = "Tokenization is fascinating, but antidisestablishmentarianism is a mouthful."
token_ids = encoding.encode(text)

print(f"{len(text)} characters -> {len(token_ids)} tokens\n")
for token_id in token_ids:
    piece = encoding.decode([token_id])
    print(f"{token_id:>7}  {piece!r}")

Notice how "Tokenization" itself gets split into `Token` + `ization`, common short words like ` is`, ` a`, ` but` stay whole single tokens (note the leading space: it's part of the token, since word boundaries matter), and the deliberately obscure word "antidisestablishmentarianism" gets broken into six separate pieces — exactly the same frequent-word-stays-whole, rare-word-gets-split behavior we saw in our tiny 8-merge example, just at a much larger scale.

Your turn to play: try encoding your own sentences above (a different language, a made-up word, some code, an emoji...) and see how they get split.

## Key takeaways

* A token is neither a word nor a character: it's a subword unit, learned from data by an algorithm like BPE, that keeps common words whole and splits rare ones into frequent pieces.
* This vocabulary is *learned*, not universal: different model providers train their own tokenizer on their own data, so the very same text can turn into a different number of tokens, and different token pieces, depending on which LLM you send it to.
* This is why LLM APIs talk about *tokens* rather than words or characters: context window limits, the `max_tokens` parameter, and pricing are all expressed in tokens — something we'll come back to in a later part.